# Class 7 Challenge — Model the Business Workflow with Data

> Client: “Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”

**Project KPI: reduce Late Delivery Rate.** Per the brief, we do not repeat retrieval or cleaning: this notebook starts from the pipeline's **silver** layer (typed, de-duplicated, system-of-record timestamps resolved). The logic proven here became `pipeline/transform.py` and `pipeline/metrics.py`.

In [1]:
import subprocess, sys
from pathlib import Path
import pandas as pd
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SILVER = BASE / "data" / "silver" / "run_date=2026-09-27"
if not SILVER.exists():   # build it with the pipeline (one command) if this is a fresh clone
    subprocess.run([sys.executable, "run_pipeline.py", "--run-date", "2026-09-27"], cwd=BASE, check=True, capture_output=True)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 60)
T = ["created_at", "promised_eta", "pickup_at_resolved", "delivered_at_resolved", "assigned_at", "reassigned_at", "estimated_pickup_at",
     "current_delivery_eta", "timestamp", "action_at", "intervention_at", "last_updated_at"]
def _load(n):
    df = pd.read_csv(SILVER / f"{n}.csv")
    for c in df.columns:
        if c in T: df[c] = pd.to_datetime(df[c])
    return df
orders, dispatch, events, tickets, actions, interventions, rstatus = (_load(n) for n in
    ["orders", "dispatch", "driver_events", "support_tickets", "customer_app_actions", "order_interventions", "restaurant_status"])
print({n: len(d) for n, d in zip(["orders","dispatch","driver_events","tickets","actions","interventions","restaurant_status"], [orders, dispatch, events, tickets, actions, interventions, rstatus])})

{'orders': 1600, 'dispatch': 1600, 'driver_events': 10035, 'tickets': 201, 'actions': 2365, 'interventions': 260, 'restaurant_status': 500}


# Challenge 1 — Reconstruct the order lifecycle
Three orders — delivered on time, delivered late, one with an intervention — with `event_time | event_type | actor | source_system`, collected from every source and **sorted by time**.

In [2]:
def timeline(order_id):
    o = orders[orders.order_id == order_id].iloc[0]; d = dispatch[dispatch.order_id == order_id]
    rows = [(o.created_at, "ORDER_CREATED", f"customer {o.customer_id}", "order_db"),
            (o.promised_eta, "PROMISED_ETA (deadline)", "ETA service", "order_db")]
    for _, r in d.iterrows():
        rows += [(r.assigned_at, "DRIVER_ASSIGNED", f"driver {r.original_driver_id}", "dispatch_api"),
                 (r.estimated_pickup_at, "ESTIMATED_PICKUP (plan)", "dispatch", "dispatch_api")]
        if pd.notna(r.reassigned_at): rows.append((r.reassigned_at, "DRIVER_REASSIGNED", f"driver {r.driver_id_current}", "dispatch_api"))
    for _, e in events[(events.order_id == order_id) & (events.type != "gps_ping")].iterrows():
        rows.append((e.timestamp, e.type.upper(), f"driver {e.driver_id}", "driver_app"))
    rows.append((None, f"GPS_PINGS x{((events.order_id == order_id) & (events.type == 'gps_ping')).sum()}", "driver", "driver_app"))
    for _, a in actions[actions.order_id == order_id].iterrows(): rows.append((a.action_at, a.action_type, "customer", "customer_app"))
    for _, t in tickets[tickets.order_id == order_id].iterrows(): rows.append((pd.to_datetime(t.created_at), f"TICKET {t.category}", "customer", "support_platform"))
    for _, i in interventions[interventions.order_id == order_id].iterrows(): rows.append((i.intervention_at, f"INTERVENTION {i.intervention_type} ({i.reason_code})", i.initiated_by, "ops_log"))
    for _, s in rstatus[rstatus.order_id == order_id].iterrows(): rows.append((s.last_updated_at, f"RESTAURANT_STATUS {s.status} (snapshot)", "restaurant", "restaurant_feed"))
    tl = pd.DataFrame(rows, columns=["event_time", "event_type", "actor", "source_system"]).sort_values("event_time", na_position="last")
    delay = (o.delivered_at_resolved - o.promised_eta).total_seconds() / 60 if pd.notna(o.delivered_at_resolved) else None
    print(f"\n=== {order_id} | status={o.final_status} | delay vs promise = {delay:.1f} min" if delay is not None else f"\n=== {order_id} | {o.final_status}")
    return tl.reset_index(drop=True)

o = orders[orders.final_status == "delivered"].assign(delay=lambda x: (x.delivered_at_resolved - x.promised_eta).dt.total_seconds() / 60)
on_time = o[(o.delay < -3) & ~o.order_id.isin(actions.order_id)].order_id.iloc[0]
late = o[(o.delay > 15) & o.order_id.isin(tickets.order_id)].order_id.iloc[0]
with_iv = o[o.order_id.isin(interventions.order_id) & (o.delay > 0) & o.order_id.isin(actions.order_id)].order_id.iloc[0]
for oid in [on_time, late, with_iv]: display(timeline(oid))


=== O00003 | status=delivered | delay vs promise = -4.4 min


,event_time,event_type,actor,source_system
0,2026-08-01 19:35:00,ORDER_CREATED,customer C0855,order_db
1,2026-08-01 19:35:00,DRIVER_ASSIGNED,driver D039,dispatch_api
2,2026-08-01 19:35:00,ASSIGNED,driver D039,driver_app
3,2026-08-01 19:48:00,RESTAURANT_STATUS preparing (snapshot),restaurant,restaurant_feed
4,2026-08-01 19:53:00,ESTIMATED_PICKUP (plan),dispatch,dispatch_api
5,2026-08-01 19:57:50,PICKED_UP,driver D039,driver_app
6,2026-08-01 20:13:28,DELIVERED,driver D039,driver_app
7,2026-08-01 20:17:52,PROMISED_ETA (deadline),ETA service,order_db
8,NaT,GPS_PINGS x3,driver,driver_app



=== O00037 | status=delivered | delay vs promise = 15.4 min


,event_time,event_type,actor,source_system
0,2026-08-09 20:18:00,ORDER_CREATED,customer C0346,order_db
1,2026-08-09 20:21:33,DRIVER_ASSIGNED,driver D076,dispatch_api
2,2026-08-09 20:21:33,ASSIGNED,driver D076,driver_app
3,2026-08-09 20:35:00,ETA_VIEWED,customer,customer_app
4,2026-08-09 20:36:00,ESTIMATED_PICKUP (plan),dispatch,dispatch_api
5,2026-08-09 20:37:00,ETA_VIEWED,customer,customer_app
6,2026-08-09 20:41:00,ETA_VIEWED,customer,customer_app
7,2026-08-09 20:50:48,PICKED_UP,driver D076,driver_app
8,2026-08-09 21:09:00,TICKET late_delivery,customer,support_platform
9,2026-08-09 21:40:18,PROMISED_ETA (deadline),ETA service,order_db



=== O00016 | status=delivered | delay vs promise = 2.5 min


,event_time,event_type,actor,source_system
0,2026-08-14 12:53:00,ORDER_CREATED,customer C0326,order_db
1,2026-08-14 12:58:56,DRIVER_ASSIGNED,driver D102,dispatch_api
2,2026-08-14 12:58:56,ASSIGNED,driver D102,driver_app
3,2026-08-14 13:04:00,INTERVENTION REFUND_OFFER (CUSTOMER_CONTACT),support,ops_log
4,2026-08-14 13:08:00,ETA_VIEWED,customer,customer_app
5,2026-08-14 13:11:00,ESTIMATED_PICKUP (plan),dispatch,dispatch_api
6,2026-08-14 13:21:47,PICKED_UP,driver D102,driver_app
7,2026-08-14 14:11:06,PROMISED_ETA (deadline),ETA service,order_db
8,2026-08-14 14:13:35,DELIVERED,driver D102,driver_app
9,NaT,GPS_PINGS x4,driver,driver_app


**What the timelines show:** in the late order, the gap between `ESTIMATED_PICKUP (plan)` and the real `PICKED_UP` is where the promise is lost, while transit looks normal. The customer's reaction (ETA views → ticket → support opened) comes *after* the damage is done — and O00037's “late_delivery” ticket is raised **31 minutes before** the stored promised ETA, so the customer was looking at a different, earlier ETA (check V22). No source records when the driver **arrived at the restaurant** or when the food was **ready** — the restaurant “status” is one snapshot, sometimes after pickup.

# Challenge 2 — Canonical project model

| Table | Grain (one row =) | Primary key | Important foreign keys |
|---|---|---|---|
| `customer` | customer | `customer_id` | — |
| `order` (entity + outcome) | order | `order_id` | `customer_id`, `restaurant_id`, `driver_id_current` |
| `order_event` (timeline) | state change of an order | (`order_id`, `event_time`, `event_type`) | `order_id` |
| `customer_interaction` | app action or ticket | `action_id` / `ticket_id` | `order_id`, `customer_id` |
| `intervention` | ops/support action on an order | `intervention_id` | `order_id` |
| `order_journey` (analytics mart) | order, with everything aggregated to it | `order_id` | all of the above |

**Why not mirror every source table?** Sources are organised by *system* (Dispatch, Driver App, Support…); the business question is organised by *workflow* (created → assigned → picked up → delivered, with reactions and interventions along the way). Mirroring 9 source tables forces every analyst to redo the joins and the system-of-record decisions (whose pickup time? which driver?). The model makes those decisions once. Relationships: customer 1–N order; order 1–N events / interactions / interventions; restaurant 1–N order; driver 1–N order.

# Challenge 3 — interaction → intervention → outcome (one row per order)
Aggregate the one-to-many tables to the order grain **before** joining.

In [3]:
act = actions.pivot_table(index="order_id", columns="action_type", values="action_id", aggfunc="count", fill_value=0)
ivs = interventions.groupby("order_id").agg(intervention_count=("intervention_id", "count"), intervention_types=("intervention_type", lambda s: ",".join(sorted(set(s)))))
tk = tickets.dropna(subset=["order_id"]).groupby("order_id").agg(ticket_count=("ticket_id", "count"))
model = (orders[["order_id", "customer_id", "final_status", "created_at", "promised_eta", "delivered_at_resolved", "flag_promise_invalid"]]
         .join(act, on="order_id").join(tk, on="order_id").join(ivs, on="order_id"))
model[["ETA_VIEWED", "SUPPORT_OPENED", "CANCEL_ATTEMPTED", "ticket_count", "intervention_count"]] = model[["ETA_VIEWED", "SUPPORT_OPENED", "CANCEL_ATTEMPTED", "ticket_count", "intervention_count"]].fillna(0).astype(int)
model["delay_min"] = (model.delivered_at_resolved - model.promised_eta).dt.total_seconds() / 60
eligible = (model.final_status == "delivered") & model.delay_min.notna() & ~model.flag_promise_invalid
model["late_flag"] = pd.Series(pd.NA, index=model.index, dtype="boolean"); model.loc[eligible, "late_flag"] = model.loc[eligible, "delay_min"] > 0
model["support_opened"] = model.SUPPORT_OPENED > 0; model["cancel_attempted"] = model.CANCEL_ATTEMPTED > 0
cols = ["order_id", "customer_id", "support_opened", "cancel_attempted", "intervention_count", "intervention_types", "final_status", "late_flag", "delay_min"]
display(model[cols].head(8))
lt = model[model.late_flag == True]
print("1. late orders with support interaction (in-app support or ticket):", int((lt.support_opened | (lt.ticket_count > 0)).sum()), "of", len(lt))
print("2. orders that received an intervention:", int((model.intervention_count > 0).sum()))
print("3. most common intervention:", interventions.intervention_type.value_counts().idxmax(), interventions.intervention_type.value_counts().to_dict())
frustrated = model[(model.support_opened | model.cancel_attempted | (model.ticket_count > 0)) & (model.intervention_count == 0)]
print("4. frustrated journeys (support/cancel/ticket) with NO intervention:", len(frustrated), "— of which late:", int((frustrated.late_flag == True).sum()))
display(frustrated[cols].sort_values("delay_min", ascending=False).head(5))

,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
0,O00001,C0168,True,False,0,NaN,delivered,True,10.816667
1,O00002,C0043,False,False,0,NaN,delivered,True,3.966667
2,O00003,C0855,False,False,0,NaN,delivered,False,-4.400000
3,O00004,C0229,False,False,0,NaN,cancelled,<NA>,NaN
4,O00005,C0040,False,False,0,NaN,delivered,True,21.016667
5,O00006,C0152,False,False,0,NaN,delivered,True,7.300000
6,O00007,C0875,False,False,0,NaN,delivered,True,4.266667
7,O00008,C0223,True,False,0,NaN,delivered,True,26.850000


1. late orders with support interaction (in-app support or ticket): 258 of 863
2. orders that received an intervention: 260
3. most common intervention: DRIVER_REASSIGNMENT {'DRIVER_REASSIGNMENT': 79, 'ETA_MESSAGE': 61, 'RESTAURANT_CONTACT': 56, 'PRIORITY_DISPATCH': 41, 'REFUND_OFFER': 23}
4. frustrated journeys (support/cancel/ticket) with NO intervention: 243 — of which late: 213


,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
193,O00194,C0771,False,False,0,NaN,delivered,True,38.333333
197,O00198,C0657,False,False,0,NaN,delivered,True,35.233333
302,O00303,C0411,False,False,0,NaN,delivered,True,33.150000
210,O00211,C0629,True,False,0,NaN,delivered,True,32.383333
1234,O01235,C0772,False,False,0,NaN,delivered,True,32.000000


# Challenge 4 — 3–5 business metrics (at least one outcome, one interaction, one intervention)

| Metric | Formula | Grain | Why it matters | Link to KPI |
|---|---|---|---|---|
| **M1 Late Delivery Rate** (outcome) | late ÷ KPI-eligible delivered orders | order | the project KPI | *is* the KPI (lagging) |
| **M2 Pickup SLA Miss Rate** (workflow) | picked up > 10 min after Dispatch's estimate ÷ eligible | order | where the time is lost; controllable | leading driver — 97% of misses end late |
| **M3 Customer Escalation Rate** (interaction) | late orders with support opened or ticket ÷ late orders | order | CX damage & support load | consequence of M1; early warning |
| **M4 Preventive Intervention Coverage** (intervention) | late orders with a non-refund intervention before promise ÷ late orders | order | are we acting in time? | controllable lever on M1 |
| **M5 ETA Drift** (diagnostic) | median(current ETA − promised ETA) | order | “ETA keeps changing” complaints | explains customer perception |

In [4]:
j = model[eligible].merge(orders[["order_id", "pickup_at_resolved"]], on="order_id").merge(dispatch[["order_id", "estimated_pickup_at", "current_delivery_eta", "assigned_at"]], on="order_id")
j["pickup_overrun"] = (j.pickup_at_resolved - j.estimated_pickup_at).dt.total_seconds() / 60
j["escalated"] = j.support_opened | (j.ticket_count > 0)
prev = interventions[(interventions.preventive == True)].order_id
L = j[j.late_flag == True]
metrics = pd.Series({
    "M1 late delivery rate %": 100 * j.late_flag.mean(),
    "M2 pickup SLA miss rate %": 100 * (j.pickup_overrun > 10).mean(),
    "M3 escalation rate among late %": 100 * L.escalated.mean(),
    "M4 preventive intervention coverage of late %": 100 * L.order_id.isin(prev).mean(),
    "M5 ETA drift (median min)": ((j.current_delivery_eta - j.promised_eta).dt.total_seconds() / 60).median(),
}).round(1)
metrics

M1 late delivery rate %                          56.5
M2 pickup SLA miss rate %                        42.0
M3 escalation rate among late %                  29.9
M4 preventive intervention coverage of late %    15.2
M5 ETA drift (median min)                         8.1
dtype: float64

# Challenge 5 — Investigate the workflow (A–E)

In [5]:
print("A. median delay with vs without customer support interaction:")
display(j.groupby("escalated").delay_min.agg(orders="count", median="median", late_rate=lambda s: round(100*(s>0).mean(),1)).round(1))
print("B. late rate with vs without intervention:")
display(j.assign(has_iv=j.intervention_count > 0).groupby("has_iv").late_flag.agg(orders="count", late_rate=lambda s: round(100*s.mean(),1)))
print("C. late rate by intervention type:")
display(interventions.merge(j[["order_id", "late_flag"]], on="order_id").groupby("intervention_type").late_flag.agg(orders="count", late_rate=lambda s: round(100*s.astype(float).mean(),1)).sort_values("late_rate"))
print("D. restaurants contributing the most late orders:")
rest = j.merge(orders[["order_id", "restaurant_id"]], on="order_id").groupby("restaurant_id").late_flag.agg(orders="count", late=lambda s: int(s.sum()), late_rate=lambda s: round(100*s.astype(float).mean(),1))
display(rest.sort_values("late", ascending=False).head(5))
E = j[j.escalated & (j.intervention_count > 0) & (j.late_flag == True)]
print("E. support interaction + intervention + still late:", len(E)); display(E[["order_id", "delay_min", "intervention_types", "ticket_count"]].sort_values("delay_min", ascending=False).head(5))

A. median delay with vs without customer support interaction:


,orders,median,late_rate
escalated,,,
False,1236,-0.2,48.9
True,292,12.2,88.4


B. late rate with vs without intervention:


,orders,late_rate
has_iv,,
False,1282,56.3
True,246,57.3


C. late rate by intervention type:


,orders,late_rate
intervention_type,,
REFUND_OFFER,22,45.5
PRIORITY_DISPATCH,38,52.6
DRIVER_REASSIGNMENT,77,55.8
RESTAURANT_CONTACT,51,56.9
ETA_MESSAGE,58,67.2


D. restaurants contributing the most late orders:


,orders,late,late_rate
restaurant_id,,,
R050,35,22,62.9
R024,26,21,80.8
R023,32,20,62.5
R004,30,20,66.7
R030,38,20,52.6


E. support interaction + intervention + still late: 46


,order_id,delay_min,intervention_types,ticket_count
697,O00737,36.283333,RESTAURANT_CONTACT,0
159,O00169,28.950000,DRIVER_REASSIGNMENT,1
73,O00077,26.550000,DRIVER_REASSIGNMENT,0
1263,O01323,22.300000,REFUND_OFFER,1
361,O00383,21.850000,DRIVER_REASSIGNMENT,0


| Q | What it tells the business | What it does NOT prove |
|---|---|---|
| **A** | Escalating customers are almost always genuinely late (median delay far higher) — support contacts are a reliable late-order signal | that support contact *causes* anything; it is a consequence |
| **B** | Intervened orders are late about as often as the rest (~57–58% vs ~56%) | that interventions don't work — ops targets at-risk orders (selection bias), no control group |
| **C** | Among preventive types PRIORITY_DISPATCH ends late least often (52.6%) and ETA_MESSAGE most (67.2%) — a message informs the customer, it does not speed the order up | a ranking of effectiveness — types are applied to different kinds of orders, and reason codes don't match types |
| **D** | Late orders are spread across all 60 restaurants (rates ~35–80%) — no single “bad restaurant” explains the KPI | restaurant blame: without a FOOD_READY / ARRIVED event we cannot separate kitchen from rider |
| **E** | A group of journeys where we engaged *and* still failed — the best sample for an ops post-mortem | why each failed; the interventions are mostly logged *before pickup*, yet pickup still slipped |

# Challenge 6 — Connect the model to the KPI

```text
PROJECT KPI   Reduce Late Delivery Rate (M1, lagging)                       <- order_db promise + delivery
  └ OUTCOME   delivered_late / delivered_on_time per order                   <- order_db, driver_app (fallback)
      └ WORKFLOW DRIVERS
          ├ assignment lag (≈ 2 min, same for late & on-time)               <- dispatch_api
          ├ FIRST MILE: pickup vs Dispatch estimate (M2)  ◄── the gap        <- dispatch_api + driver_app
          │     ├ restaurant readiness ?  (no FOOD_READY event)              <- restaurant feed = snapshot only
          │     └ driver arrival ?        (no ARRIVED event, GPS every 13 min) <- driver_app
          └ transit (traffic/weather, +3.5 min)                              <- driver_app, order_db buckets
      └ CUSTOMER REACTION  ETA views, support opened, tickets (M3), ETA drift (M5)  <- customer_app, support
      └ INTERVENTIONS  reassignment, restaurant contact, priority dispatch (M4)     <- ops log (≠ dispatch!)
```

1. **Directly controllable by operations:** M2 (pickup SLA — dispatch planning + restaurant hand-off) and M4 (intervene before the promise).
2. **Outcomes:** M1 (and M3 as the customer-side consequence).
3. **Missing event that limits the model most:** `DRIVER_ARRIVED_AT_RESTAURANT` (+ a real `FOOD_READY` history) — without it the first-mile delay cannot be attributed to restaurant vs driver.
4. **Instrument next:** driver “Arrived” button / geofence event, restaurant “Food ready” event history, the ETA actually shown to the customer, and a real distance (stop defaulting to 18 km).

**Final deliverable:** entities = customer, order, restaurant, driver, ticket, intervention · key events = created, assigned, reassigned, picked up, delivered, app actions, tickets, interventions · relationships = order-centric 1-to-N · metrics M1–M5 · KPI linkage above · **limitation:** the model can say *where* time is lost (first mile) but not *who* loses it.